# 🤖 Week 3 Assignment — RAG-Powered Console Chatbot
### WnCC Machine Learning Learner Space 2026

---

This week's assignment has three parts of increasing difficulty.

**Part A — Concept Check:** Short questions testing your understanding of the week's theory.
**Part B — Core Implementation:** Build a complete RAG pipeline from scratch and wire it to an LLM.
**Part C — The Challenge:** Make your chatbot smarter with multi-turn memory and source citation.

**Rules:**
- All `# TODO` blocks must be completed. Read the docstring above each one carefully.
- Run every **Sanity Check** cell after completing a TODO. Fix failures before moving on.
- Part A answers go in the Markdown cells provided — write proper explanations, not one-liners.
- Parts B and C are graded on correctness (sanity checks), code quality, and your reflection.

---
**Run the setup cell first.**

In [ ]:
# ========== SETUP (run this first) ==========
!pip install "transformers<5.0.0" torch faiss-cpu sentence-transformers -q

# Authenticate with HuggingFace
# Get your free token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
HF_TOKEN = "hf_*doNGrOWo*DNcXd*BOcEkXm*ANJqcMxTza"   # <-- paste your token here
login(HF_TOKEN)

import torch
import numpy as np
import faiss
import textwrap
from transformers import pipeline

print('Setup complete!')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Setup complete!
Device: GPU


---
## Part A — Concept Check

Answer each question in the Markdown cell below it. **2–4 sentences each.** Be precise.

These are not trick questions — they test whether you read the material, not whether you can Google.

---
### A1. Self-Attention

The self-attention formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In your own words: what does the matrix $QK^T$ represent before the softmax is applied? What does it become after the softmax? And what does multiplying by $V$ finally produce?

**Your answer :**

Before Softmax is applied, this multiplication represents the raw score matrix. After applying softmax, the matrix gets transformed into probabilities. Multiplying by  V  finally produces a weighted sum of the value vectors.

---
### A2. Encoder vs Decoder

Your manager asks you to build two systems:
1. A tool that reads customer support tickets and tags them as `billing`, `technical`, or `general`.
2. A tool that automatically drafts a reply to each ticket.

Which Transformer family (encoder-only, decoder-only, or encoder-decoder) would you use for each, and why? Be specific about what architectural property makes each one suited to its task.

**Your answer:**

1. Support Ticket Classification -> Encoder-only, because to categorize a text, the model needs to understand the complete context of the message. Encoder-only models look at the entire sequence at once—every token attends to every other token simultaneously, both to its left and its right. Because it doesn't need to predict future words, it isn't restricted by causal masking. This allows it to build a deep, holistic representation of the entire ticket, making it highly accurate for extracting features and assigning a single classification label.

2. Drafting Ticket Replies ->  Encoder-Decoder, because to draft a reply, the system must generate new text. This requires a Decoder. Decoders use causal masking, which hides future tokens, forcing the model to generate text strictly left-to-right one word at a time. The Encoder uses unmasked attention to fully comprehend the user's original ticket. The Decoder uses masked attention to write the reply, while utilizing cross-attention to constantly look back at the Encoder's representation of the original ticket to ensure the reply stays highly relevant to the customer's problem.

---
### A3. RLHF

Explain in your own words why RLHF uses human *comparisons* ("which of these two responses is better?") rather than human-written *ideal responses* ("write the perfect answer to this question"). What practical problem does this solve?

**Your answer:**

For many prompts (like "write a polite rejection email" or "tell a funny joke"), there is no single "perfect" answer. Forcing a human to write an ideal response restricts the model to that specific annotator's writing style. Ranking allows the reward model to map out general human preferences (tone, safety, helpfulness) across a wide distribution of acceptable answers.


---
### A4. RAG vs Fine-tuning

A startup has 10,000 internal company documents (policies, FAQs, product specs) and wants to build a Q&A bot over them. The documents are updated weekly.

They are deciding between: (a) fine-tuning an LLM on all the documents, or (b) building a RAG system.

Make the case for RAG. Give at least two concrete reasons why it is better suited to this scenario.

**Your answer:**

1. With RAG, updating the system's knowledge requires zero model retraining. It is simply a database operation. You just delete the vector embeddings of the old documents and add the embeddings of the new ones. The LLM instantly reads from the most current information.

2. RAG decouples knowledge from language generation. Because the system retrieves exact text chunks from the company's database and places them directly into the prompt, the model can provide accurate citations.


---
## Part B — Core Implementation

You will build a **RAG-powered chatbot** in stages:

```
Documents → Chunk → Embed → FAISS Index   (offline, build once)
                                  │
User question → Embed → Retrieve top-k chunks → Build prompt → LLM → Answer
```

The document corpus, embedding model, and LLM pipeline are all provided.
**Your job is to implement each stage of the pipeline.**

In [ ]:
# ========== PROVIDED: Document Corpus ==========
# A set of short ML-focused documents. Your chatbot will answer questions about these.
DOCUMENTS = [
    """The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Google Brain. It replaced recurrent neural networks with self-attention
    mechanisms, allowing the model to process all tokens in parallel rather than sequentially.
    This enabled training on much larger datasets and dramatically improved performance on
    sequence-to-sequence tasks like translation and summarisation.""",

    """BERT (Bidirectional Encoder Representations from Transformers) is an encoder-only
    Transformer introduced by Google in 2018. It is trained using Masked Language Modelling,
    where 15% of input tokens are randomly masked and the model must predict them using
    both left and right context simultaneously. BERT is primarily used for text classification,
    named entity recognition, and extractive question answering.""",

    """GPT (Generative Pre-trained Transformer) is a decoder-only Transformer developed by
    OpenAI. It is trained on next-token prediction: given a sequence of tokens, predict the
    next one. GPT models use causal attention, meaning each token can only attend to previous
    tokens. This makes them naturally suited for text generation tasks. GPT-4, released in
    2023, is estimated to have around 1 trillion parameters.""",

    """RLHF (Reinforcement Learning from Human Feedback) is the training technique used to
    align language models with human preferences. It has three stages: supervised fine-tuning
    on instruction-following examples, training a reward model on human comparisons of
    model outputs, and using PPO (Proximal Policy Optimisation) to fine-tune the language
    model to maximise the reward model's scores. ChatGPT and Claude were both trained with RLHF.""",

    """RAG (Retrieval-Augmented Generation) is a technique that combines a language model with
    an external knowledge retrieval system. Documents are chunked, embedded using a sentence
    encoder, and stored in a vector database. At inference time, the user's query is embedded,
    the most similar document chunks are retrieved, and these chunks are injected into the
    LLM's prompt as context. This allows the model to answer questions about documents it was
    never trained on, without the hallucination risk of relying solely on parametric memory.""",

    """Word2Vec is a family of models introduced by Mikolov et al. at Google in 2013 for
    learning dense word embeddings. The two architectures are Skip-gram (predicts context
    words given a centre word) and CBOW (predicts a centre word given its context words).
    Words with similar meanings cluster together in the embedding space, enabling vector
    arithmetic such as: king - man + woman ≈ queen. Word2Vec embeddings are static —
    each word has one fixed vector regardless of context.""",

    """Few-shot prompting is a technique where 2-5 examples of a task are included directly
    in the prompt, allowing the LLM to infer the task format without any weight updates.
    Chain-of-Thought (CoT) prompting extends this by including step-by-step reasoning in
    the examples, which dramatically improves performance on arithmetic, logical, and
    multi-step reasoning tasks. Simply appending 'Let's think step by step' to a prompt
    enables zero-shot CoT reasoning.""",

    """Vector databases store high-dimensional embeddings and support efficient approximate
    nearest-neighbour search. Given a query vector, they return the K stored vectors with
    the highest cosine similarity or dot product. Popular options include FAISS (Facebook
    AI Similarity Search, open-source), Pinecone (managed cloud service), Chroma (lightweight,
    local), and Weaviate (open-source with a GraphQL API). FAISS uses techniques like
    product quantisation and inverted file indices to search billions of vectors in milliseconds.""",
]

print(f'Corpus: {len(DOCUMENTS)} documents loaded.')

Corpus: 8 documents loaded.


In [ ]:
# ========== PROVIDED: Models ==========
# Do NOT change these — they are fixed for the assignment.

print('Loading embedding model...')
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = pipeline(
    'feature-extraction',
    model=EMBED_MODEL,
    tokenizer=EMBED_MODEL,
    device=-1,   # -1 = CPU; change to 0 for GPU if available
)
EMBED_DIM = 384
print(f'Embedding model loaded. Output dim: {EMBED_DIM}')

print('Loading text generation model...')
generator = pipeline(
    'text2text-generation',
    model='google/flan-t5-base',
    device=-1,
)
print('Generation model loaded.')

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Generation model loaded.


---
### B1 — Embed a Single Text  `[TODO]`

The embedding pipeline returns a nested list. Your job is to:
1. Run the text through `embedder`
2. Mean-pool over the token dimension (average across all token vectors)
3. Return a 1D numpy array of shape `(EMBED_DIM,)` with dtype `float32`

In [ ]:
def embed_text(text: str) -> np.ndarray:
    """
    Embed a single string into a dense vector.
    Args:
        text: input string
    Returns:
        numpy array of shape (EMBED_DIM,), dtype float32

    TODO:
      1. raw = embedder(text)             # returns a nested list: [[[f, f, f, ...], ...], ...]
      2. Convert raw[0] to a 2D numpy array of shape (num_tokens, EMBED_DIM)
      3. Mean-pool along axis 0 -> shape (EMBED_DIM,)
      4. Return as float32
    """
    # YOUR CODE HERE
    # 1. Get raw embeddings. Output shape: (1, num_tokens, EMBED_DIM)
    raw = embedder(text)

    # 2. Extract the first sequence and convert to 2D numpy array
    # token_embeddings shape: (num_tokens, EMBED_DIM)
    token_embeddings = np.array(raw[0])

    # 3. Mean-pool along axis 0 to average the token embeddings
    # pooled_vector shape: (EMBED_DIM,)
    pooled_vector = np.mean(token_embeddings, axis=0)

    # 4. Ensure the output is explicitly cast to float32
    return pooled_vector.astype(np.float32)
    pass


# ===== Sanity Check =====
vec = embed_text('Hello world')
assert vec is not None, 'embed_text() returned None'
assert isinstance(vec, np.ndarray), f'Expected np.ndarray, got {type(vec)}'
assert vec.shape == (EMBED_DIM,), f'Expected shape ({EMBED_DIM},), got {vec.shape}'
assert vec.dtype == np.float32, f'Expected float32, got {vec.dtype}'
print(f'PASS: embed_text() works! Shape: {vec.shape}')

PASS: embed_text() works! Shape: (384,)


---
### B2 — Chunk Documents  `[TODO]`

Split a list of long documents into smaller, overlapping chunks.
This ensures no single chunk is too long for the model and that
information at chunk boundaries is not lost.

In [ ]:
def chunk_documents(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[str]:
    """
    Split each document into overlapping character-level chunks.

    Args:
        documents: list of document strings
        chunk_size: target size of each chunk in characters
        overlap: number of characters to overlap between consecutive chunks

    Returns:
        flat list of chunk strings

    TODO:
      For each document:
        Use a sliding window of size chunk_size with step (chunk_size - overlap).
        i.e., start positions: 0, chunk_size-overlap, 2*(chunk_size-overlap), ...
        Each chunk is doc[i : i + chunk_size].
        Strip whitespace from each chunk and skip empty ones.
      Return the flat list of all chunks across all documents.
    """
    # YOUR CODE HERE
    all_chunks = []

    # Calculate how far to advance the window each time
    step = chunk_size - overlap

    # Safety check to prevent infinite loops if overlap >= chunk_size
    if step <= 0:
        raise ValueError("overlap must be less than chunk_size")

    for doc in documents:
        # Slide a window across the document from 0 to its length
        for i in range(0, len(doc), step):
            # Extract the chunk and strip surrounding whitespace
            chunk = doc[i : i + chunk_size].strip()

            # Only keep the chunk if it's not empty
            if chunk:
                all_chunks.append(chunk)

    return all_chunks
    pass


# ===== Sanity Check =====
test_chunks = chunk_documents(['a' * 500], chunk_size=200, overlap=40)
assert test_chunks is not None, 'chunk_documents() returned None'
assert len(test_chunks) > 1, 'Should produce multiple chunks for a 500-char doc'
assert all(len(c) <= 200 for c in test_chunks), 'A chunk exceeded chunk_size'
print(f'PASS: chunk_documents() works! {len(test_chunks)} chunks from 500-char doc')

chunks = chunk_documents(DOCUMENTS)
print(f'Full corpus: {len(DOCUMENTS)} documents -> {len(chunks)} chunks')
print(f'Sample chunk: "{chunks[0][:120]}..."')

PASS: chunk_documents() works! 4 chunks from 500-char doc
Full corpus: 8 documents -> 27 chunks
Sample chunk: "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Googl..."


---
### B3 — Build the FAISS Index  `[TODO]`

Embed all chunks and store them in a FAISS index for fast nearest-neighbour retrieval.

In [ ]:
def build_index(chunks: list[str]) -> tuple[faiss.Index, np.ndarray]:
    """
    Embed all chunks and build a FAISS inner-product index.

    Args:
        chunks: list of text chunks
    Returns:
        (index, embeddings) where:
          - index is a faiss.IndexFlatIP ready for search
          - embeddings is a float32 array of shape (len(chunks), EMBED_DIM)

    TODO:
      1. Embed each chunk using embed_text() -> stack into shape (N, EMBED_DIM)
         Hint: np.array([embed_text(c) for c in chunks])
      2. L2-normalise the embeddings in-place:
         faiss.normalize_L2(embeddings)   # modifies in-place
         (after normalisation, inner product = cosine similarity)
      3. Create index = faiss.IndexFlatIP(EMBED_DIM)
      4. Add embeddings to the index: index.add(embeddings)
      5. Return (index, embeddings)
    """
    # YOUR CODE HERE
    # 1. Embed each chunk and stack them into a 2D numpy array of shape (N, EMBED_DIM)
    embeddings = np.array([embed_text(c) for c in chunks])

    # 2. L2-normalize the embeddings in-place
    # When vectors are L2-normalized, the inner product is mathematically
    # equivalent to cosine similarity.
    faiss.normalize_L2(embeddings)

    # 3. Create the FAISS index for Inner Product
    index = faiss.IndexFlatIP(EMBED_DIM)

    # 4. Add the normalized embeddings to our index
    index.add(embeddings)

    # 5. Return both the search index and the raw embedding matrix
    return index, embeddings
    pass


print('Building index (this takes ~30 seconds on CPU)...')
index, embeddings = build_index(chunks)

# ===== Sanity Check =====
assert index is not None, 'build_index() returned None'
assert index.ntotal == len(chunks), f'Expected {len(chunks)} vectors in index, got {index.ntotal}'
assert embeddings.shape == (len(chunks), EMBED_DIM)
print(f'PASS: Index built! {index.ntotal} vectors of dim {EMBED_DIM}')

Building index (this takes ~30 seconds on CPU)...
PASS: Index built! 27 vectors of dim 384


---
### B4 — Retrieve Relevant Chunks  `[TODO]`

Given a question, embed it and find the most semantically similar chunks.

In [ ]:
def retrieve(
    question: str,
    index: faiss.Index,
    chunks: list[str],
    k: int = 3,
) -> list[tuple[str, float]]:
    """
    Retrieve the top-k most relevant chunks for a question.

    Args:
        question: user's query string
        index: FAISS index containing chunk embeddings
        chunks: original list of chunk strings (same order as index)
        k: number of chunks to retrieve
    Returns:
        list of (chunk_text, similarity_score) tuples, sorted by score descending

    TODO:
      1. Embed the question using embed_text() -> shape (EMBED_DIM,)
      2. Reshape to (1, EMBED_DIM) and convert to float32
      3. L2-normalise: faiss.normalize_L2(query_vec)
      4. Search: scores, indices = index.search(query_vec, k)
         scores and indices are shape (1, k) — take [0] to get 1D arrays
      5. Return [(chunks[idx], score) for idx, score in zip(indices[0], scores[0])]
    """
    # YOUR CODE HERE
    # 1. Embed the question -> shape (EMBED_DIM,)
    query_vec = embed_text(question)

    # 2. Reshape to a 2D row vector (1, EMBED_DIM) and ensure it's float32
    query_vec = np.array([query_vec], dtype=np.float32)

    # 3. L2-normalize the query vector in-place
    faiss.normalize_L2(query_vec)

    # 4. Search the FAISS index
    # scores and indices will both have the shape (1, k)
    scores, indices = index.search(query_vec, k)

    # 5. Extract results by mapping the returned indices back to the original text chunks
    # We use .item() to ensure the score is a native Python float instead of a numpy type
    results = [
        (chunks[idx], float(score))
        for idx, score in zip(indices[0], scores[0])
    ]

    return results
    pass


# ===== Sanity Check =====
test_results = retrieve('What is BERT?', index, chunks, k=3)
assert test_results is not None, 'retrieve() returned None'
assert len(test_results) == 3, f'Expected 3 results, got {len(test_results)}'
assert isinstance(test_results[0][0], str), 'First element of each tuple should be a string'
assert isinstance(test_results[0][1], float), 'Second element of each tuple should be a float'
print('PASS: retrieve() works!')
print('\nTop 3 chunks for "What is BERT?":')
for i, (chunk, score) in enumerate(test_results):
    print(f'\n[{i+1}] Score: {score:.4f}')
    print(textwrap.fill(chunk[:200], width=80))

PASS: retrieve() works!

Top 3 chunks for "What is BERT?":

[1] Score: 0.6343
BERT (Bidirectional Encoder Representations from Transformers) is an encoder-
only     Transformer introduced by Google in 2018. It is trained using Masked
Language Modelling,     where 15% of input to

[2] Score: 0.5801
age Modelling,     where 15% of input tokens are randomly masked and the model
must predict them using     both left and right context simultaneously. BERT is
primarily used for text classification,

[3] Score: 0.2323
isation) to fine-tune the language     model to maximise the reward model's
scores. ChatGPT and Claude were both trained with RLHF.


---
### B5 — Build the RAG Prompt  `[TODO]`

Format the retrieved chunks and the user's question into a prompt the LLM can use.

In [ ]:
def build_prompt(question: str, context_chunks: list[tuple[str, float]]) -> str:
    """
    Build an LLM prompt from a question and retrieved context chunks.

    Args:
        question: user's question
        context_chunks: list of (chunk_text, score) tuples from retrieve()
    Returns:
        formatted prompt string

    TODO:
      Build a prompt with this structure:

      Answer the question using only the context below.
      If the answer is not in the context, say "I don't know".

      Context:
      [Source 1]: <chunk text>

      [Source 2]: <chunk text>

      [Source 3]: <chunk text>

      Question: <question>
      Answer:

    Each source should be on its own line, separated by a blank line.
    """
    # YOUR CODE HERE
    # Initialize the prompt with the system instructions and header
    lines = [
        "Answer the question using only the context below.",
        "If the answer is not in the context, say 'I don't know'.",
        "",
        "Context:"
    ]

    # Iterate through the chunks, enumerating from 1 for the source numbers
    for i, (chunk_text, score) in enumerate(context_chunks, start=1):
        lines.append(f"[Source {i}]: {chunk_text}")
        lines.append("")  # Adds the required blank line between sources

    # Append the final question and answer cue
    lines.append(f"Question: {question}")
    lines.append("Answer:")

    # Join everything together with newline characters
    return "\n".join(lines)
    pass


# ===== Sanity Check =====
test_chunks_for_prompt = [('Transformers were introduced in 2017.', 0.95),
                           ('BERT is an encoder model.', 0.80)]
prompt = build_prompt('When were Transformers introduced?', test_chunks_for_prompt)
assert prompt is not None, 'build_prompt() returned None'
assert 'Source 1' in prompt, 'Prompt should contain [Source 1]'
assert 'Source 2' in prompt, 'Prompt should contain [Source 2]'
assert 'When were Transformers introduced?' in prompt, 'Prompt must contain the question'
assert 'Answer:' in prompt, 'Prompt must end with Answer:'
print('PASS: build_prompt() works!')
print('\nSample prompt:')
print('-' * 60)
print(prompt)
print('-' * 60)

PASS: build_prompt() works!

Sample prompt:
------------------------------------------------------------
Answer the question using only the context below.
If the answer is not in the context, say 'I don't know'.

Context:
[Source 1]: Transformers were introduced in 2017.

[Source 2]: BERT is an encoder model.

Question: When were Transformers introduced?
Answer:
------------------------------------------------------------


---
### B6 — Generate an Answer  `[TODO]`

Pass the prompt to the LLM and extract the generated answer.

In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 150) -> str:
    """
    Generate an answer from the LLM given a RAG prompt.

    Args:
        prompt: the formatted RAG prompt from build_prompt()
        max_new_tokens: maximum tokens to generate
    Returns:
        the generated answer string (stripped)

    TODO:
      1. Call generator(prompt, max_new_tokens=max_new_tokens)
         generator returns a list of dicts; get result[0]['generated_text']
      2. Strip whitespace and return the answer string
    """
    # YOUR CODE HERE
    # 1. Call the Hugging Face generation pipeline.
    # We pass max_new_tokens to cap the output length.
    result = generator(prompt, max_new_tokens=max_new_tokens)

    # 2. The pipeline returns a list of dictionaries.
    # We extract the generated string and strip any lingering whitespace.
    answer = result[0]['generated_text'].strip()

    return answer
    pass


# ===== Sanity Check =====
test_prompt = 'Answer in one sentence. What is 2 + 2? Answer:'
answer = generate_answer(test_prompt, max_new_tokens=20)
assert answer is not None, 'generate_answer() returned None'
assert isinstance(answer, str), f'Expected str, got {type(answer)}'
assert len(answer) > 0, 'Answer is empty'
print(f'PASS: generate_answer() works! Sample output: "{answer}"')

PASS: generate_answer() works! Sample output: "2 + 2"


---
### B7 — The Full RAG Pipeline  `[TODO]`

Wire together all the pieces you've built into one function.

In [ ]:
def rag_answer(question: str, index: faiss.Index, chunks: list[str], k: int = 3) -> str:
    """
    Full RAG pipeline: question -> retrieve -> prompt -> generate -> answer.

    TODO:
      1. Call retrieve() to get the top-k chunks
      2. Call build_prompt() with the question and retrieved chunks
      3. Call generate_answer() with the prompt
      4. Return the answer string
    """
    # YOUR CODE HERE
    # 1. Retrieve the most relevant context chunks using your FAISS index
    context_chunks = retrieve(question, index, chunks, k)

    # 2. Format those chunks and the user's question into a strict prompt
    prompt = build_prompt(question, context_chunks)

    # 3. Pass the formatted prompt to the LLM to generate the final response
    answer = generate_answer(prompt)

    # 4. Return the generated text
    return answer
    pass


# ===== Sanity Check =====
test_questions = [
    'What is BERT and what is it used for?',
    'How does RLHF work?',
    'What is the difference between Word2Vec and Transformer embeddings?',
]

print('========== RAG PIPELINE TEST ==========')
for q in test_questions:
    answer = rag_answer(q, index, chunks)
    assert answer is not None and len(answer) > 0, f'Got empty answer for: {q}'
    print(f'\nQ: {q}')
    print(f'A: {textwrap.fill(answer, width=80)}')

print('\nPASS: Full RAG pipeline works!')

========== RAG PIPELINE TEST ==========

Q: What is BERT and what is it used for?
A: BERT is primarily used for text classification, [Source 2]: age Modelling, where
15% of input tokens are randomly masked and the model must predict them using
both left and right context simultaneously. BERT is primarily used for text
classification, [Source 3]: RAG (Retrieval-Augmented Generation) is a technique
that combines a language model with an external knowledge retrieval system.
Documents are chunked, embedded using a sentence encoder, and st

Q: How does RLHF work?
A: It has three stages: supervised fine-tuning on instruction-fol [Source 2]:
isation) to fine-tune the language model to maximise the reward model's scores

Q: What is the difference between Word2Vec and Transformer embeddings?
A: Word2Vec embeddings are static — each word has one fixed vector regardless of
context.

PASS: Full RAG pipeline works!


---
### B8 — The Console Chatbot  `[TODO]`

Now build the interactive loop. This is the payoff — put it all together.

In [ ]:
def run_chatbot(index: faiss.Index, chunks: list[str]) -> None:
    """
    Run an interactive console chatbot loop.

    TODO:
      Implement a loop that:
      1. Prints a welcome message
      2. Repeatedly:
         a. Prompts the user for input with: question = input('You: ').strip()
         b. If question is empty, continue (ask again)
         c. If question.lower() is 'quit' or 'exit', print a goodbye message and break
         d. Otherwise:
            - Print 'Bot: Thinking...' so the user knows it is working
            - Call rag_answer(question, index, chunks)
            - Print 'Bot: <answer>'
            - Print a blank line for readability

    Example interaction:
    ─────────────────────────────────────
    ML Chatbot (type 'quit' to exit)

    You: What is a Transformer?
    Bot: Thinking...
    Bot: The Transformer architecture was introduced in 2017...

    You: quit
    Bot: Goodbye!
    ─────────────────────────────────────
    """
    # YOUR CODE HERE
    # 1. Print a welcome message
    print("ML Chatbot (type 'quit' to exit)\n")

    # 2. Repeatedly prompt for input
    while True:
        # a. Prompt the user and strip leading/trailing whitespace
        question = input('You: ').strip()

        # b. If the user just pressed Enter, ask again
        if not question:
            continue

        # c. Check for exit commands
        if question.lower() in ['quit', 'exit']:
            print("Bot: Goodbye!")
            break

        # d. Process the valid question
        print("Bot: Thinking...")
        answer = rag_answer(question, index, chunks)
        print(f"Bot: {answer}")
        print() # Print a blank line for readability
    pass


# Run the chatbot!
# (In Colab, this will create an interactive input box)
run_chatbot(index, chunks)

ML Chatbot (type 'quit' to exit)

You: What is the weather today?
Bot: Thinking...
Bot: I don't know

You: quit
Bot: Goodbye!


---
## Part C — The Challenge

Choose **one** of the two extensions below. Both are open-ended — there is no single right answer. You are graded on your approach, code quality, and reflection.

---
### Option 1 — Multi-Turn Memory

Right now, each question is answered independently — the chatbot has no memory of previous turns.

**Task:** Modify `run_chatbot()` to maintain a conversation history, and include the last N turns in the prompt so the bot can answer follow-up questions coherently.

Example:
```
You: What is BERT?
Bot: BERT is an encoder-only Transformer...

You: What was it trained on?    ← requires memory of previous turn to answer
Bot: BERT was trained using Masked Language Modelling...
```

**Hint:** Modify `build_prompt()` to accept an optional `history: list[tuple[str,str]]` parameter (list of (question, answer) pairs) and prepend it to the context.

In [ ]:
# ===== Option 1: Multi-Turn Memory =====

def build_prompt_with_history(
    question: str,
    context_chunks: list[tuple[str, float]],
    history: list[tuple[str, str]],
    max_history: int = 3,
) -> str:
    """
    Build a RAG prompt that includes recent conversation history.

    TODO:
      Include up to `max_history` recent (question, answer) pairs from history
      in the prompt, before the retrieved context.
      Format them clearly so the model understands they are prior turns.
    """
    # YOUR CODE HERE
    pass


def run_chatbot_with_memory(index: faiss.Index, chunks: list[str]) -> None:
    """
    TODO: Modify the chatbot loop to:
      - Maintain a history list of (question, answer) tuples
      - Use build_prompt_with_history() instead of build_prompt()
      - Append each (question, answer) pair to history after each turn
    """
    # YOUR CODE HERE
    pass


run_chatbot_with_memory(index, chunks)

---
### Option 2 — Source Citation

Right now, the chatbot gives an answer but cites no sources. Users have no way to verify the answer.

**Task:** After each answer, print the top retrieved source chunks (with their similarity scores) so the user can see *where* the answer came from.

**Bonus:** Track which document each chunk came from (add a `doc_id` to each chunk during chunking) and display `[Source: Document 3]` alongside each chunk.

In [ ]:
# ===== Option 2: Source Citation =====

def chunk_documents_with_ids(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[dict]:
    """
    Like chunk_documents(), but returns a list of dicts:
    {'text': str, 'doc_id': int, 'chunk_id': int}

    TODO: Re-implement chunking to track which document each chunk came from.
    """
    # YOUR CODE HERE
    all_chunks = []
    step = chunk_size - overlap

    if step <= 0:
        raise ValueError("overlap must be less than chunk_size")

    chunk_id = 0
    # enumerate provides the doc_id automatically starting from 0
    for doc_id, doc in enumerate(documents):
        # Slide window across the current document
        for i in range(0, len(doc), step):
            chunk_text = doc[i : i + chunk_size].strip()

            if chunk_text:
                all_chunks.append({
                    'text': chunk_text,
                    'doc_id': doc_id,
                    'chunk_id': chunk_id
                })
                chunk_id += 1

    return all_chunks
    pass


def run_chatbot_with_citations(index: faiss.Index, chunks: list[str]) -> None:
    """
    TODO: Modify the chatbot loop to print the retrieved source chunks
    and their similarity scores after each answer.
    Format them clearly, e.g.:

      Sources:
      [1] (score: 0.87) BERT (Bidirectional Encoder Representations...
      [2] (score: 0.73) The Transformer architecture was introduced...
    """
    # YOUR CODE HERE
    print("ML Chatbot (type 'quit' to exit)\n")

    # Helper check: determine if chunks are dicts (Bonus) or raw strings
    is_dict_format = len(chunks) > 0 and isinstance(chunks[0], dict)

    # Extract just the text strings to pass into the retrieve() and rag_answer()
    # functions you built previously, so they don't break.
    text_chunks = [c['text'] for c in chunks] if is_dict_format else chunks

    while True:
        question = input('You: ').strip()

        if not question:
            continue

        if question.lower() in ['quit', 'exit']:
            print("Bot: Goodbye!")
            break

        print("Bot: Thinking...")

        # Generate the main answer
        answer = rag_answer(question, index, text_chunks)
        print(f"Bot: {answer}\n")

        # Retrieve the context chunks to display the citations and scores
        retrieved = retrieve(question, index, text_chunks, k=3)

        print("  Sources:")
        for i, (chunk_text, score) in enumerate(retrieved, start=1):
            doc_info = ""

            # Bonus: Find the original doc_id by matching the text
            if is_dict_format:
                for c_dict in chunks:
                    if c_dict['text'] == chunk_text:
                        doc_info = f"[Source: Document {c_dict['doc_id']}] "
                        break

            # Flatten and truncate the text so it doesn't clutter the console
            display_text = chunk_text.replace('\n', ' ')
            if len(display_text) > 75:
                display_text = display_text[:72] + "..."

            print(f"  [{i}] (score: {score:.2f}) {doc_info}{display_text}")

        print() # Blank line for readability
    pass


run_chatbot_with_citations(index, chunks)

ML Chatbot (type 'quit' to exit)

You: quit
Bot: Goodbye!


---
## Reflection (Required)

Fill in all five answers before submitting.

1. **Which chunks did your system retrieve for the question "How was ChatGPT trained?"? Were they the right ones? If not, why do you think the retrieval failed?**  

  Ans : System retrieves the following chunks : Sources:
    [1] (score: 0.63) isation) to fine-tune the language     model to maximise the reward mode...
    [2] (score: 0.38) GPT (Generative Pre-trained Transformer) is a decoder-only Transformer d...
    [3] (score: 0.30) ns, predict the     next one. GPT models use causal attention, meaning e...
    It is likely because semantic search struggles with exact entity matching if the context is sparse.

2. **Try asking your bot a question whose answer is NOT in the corpus (e.g. 'What is the weather today?'). What does it say? Is this the behaviour you want? How would you fix it?**  
   
  Ans : It replied "I don't know", because of the strict prompt instruction we gave to it (If the answer is not in the context, say "I don't know"). Yes, this was the desired behavior for our RAG system to prevent hallucinations. If it did hallucinate an answer, I would fix it by either lowering the generation model's 'temperature' to 0 to make it more deterministic, or by making the system prompt even more strict, like 'Do Not use outside knowledge under any circumstances'.

3. **What is one concrete limitation of your current chunking strategy? How would you improve it?**  
   
   Ans : Our current chunking strategy is purely mathematical — it splits text based strictly on character count (chunk_size=200). This means it will truncate words, sentences, or core concepts completely in half (e.g., splitting "virat kohli" into "vir" at the end of chunk 1, and "at kohli" at the start of chunk 2). This destroys the semantic meaning of that sentence. I would implement a recursive or natural-language-aware chunker (like LangChain's RecursiveCharacterTextSplitter). Instead of blindly cutting at 200 characters, it tries to split at natural boundaries first.


4. **Which Part C option did you implement? Describe your approach in 3–4 sentences and one thing you would do differently with more time.**  

   Ans : I implemented Option 2: Source Citation. I modified the chunking function to inject a dictionary containing a unique doc_id alongside the chunk text. Then, I updated the chatbot loop to intercept the retrieve() results and print out the top 3 chunks, their cosine similarity scores, and their originating document ID. With more time, I would map the doc_id back to the actual filename or document title instead of just an integer index, making the citations much more human-readable.

5. **You have now built systems using Bag-of-Words (Week 2) and vector embeddings + retrieval (Week 3). In your own words, what is the fundamental difference between how TF-IDF retrieval and embedding-based retrieval find relevant documents?**  

   Ans :
   1. TF-IDF relies on exact keyword matching. It counts the frequency of specific words and penalizes overly common ones. If you search for "puppy," TF-IDF will completely miss a highly relevant document that only uses the phrase "young dog" because the exact characters do not match.
   2. Embeddings are semantic. They convert text into dense, high-dimensional vectors where mathematical distance represents meaning. The embedding model understands the underlying concepts, so it places "puppy" and "young dog" very close together in the vector space. It retrieves documents based on the intent and context of your question, not just the specific words you typed.

---
**Submit your completed notebook to the WnCC submission form. Well done — you've built a RAG chatbot from scratch. 🚀**